# ПЗ 7 — Распознавание через LLM (Qwen / Gemini)

**Задача:** читать кадры из Drive, описать сцену через LLM API, сохранить JSON в Drive.

**Стек:** `openai` (Qwen), `google-generativeai` (Gemini), `fastapi`

In [ ]:
!pip install openai google-generativeai fastapi uvicorn python-multipart -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'Найдено кадров: {len(frames)}')

In [ ]:
# === ВСТАВЬТЕ ВАШИ API КЛЮЧИ ===
GEMINI_API_KEY = 'ВАШИ_КЛЮЧ_GEMINI'  # https://aistudio.google.com/app/apikey
QWEN_API_KEY   = 'ВАШИ_КЛЮЧ_QWEN'   # https://dashscope.console.aliyun.com

## Вариант A — Google Gemini

In [ ]:
import json
import google.generativeai as genai
from PIL import Image
from tqdm.notebook import tqdm

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

def describe_frame_gemini(image_path: str) -> dict:
    img = Image.open(image_path)
    prompt = 'Опиши объекты на кадре. Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'
    resp = gemini.generate_content([img, prompt])
    try:
        text = resp.text.strip().lstrip('```json').rstrip('```')
        return json.loads(text)
    except Exception:
        return {'raw': resp.text}

# Прогнать первые 10 кадров
llm_results = []
for fname in tqdm(frames[:10], desc='Gemini'):
    res = describe_frame_gemini(f'{FRAMES_DIR}/{fname}')
    res['frame'] = fname
    llm_results.append(res)
    print(f'{fname}: {res}')

OUT = f'{RESULTS_DIR}/llm_detections.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(llm_results, f, ensure_ascii=False, indent=2)
print(f'Сохранено: {OUT}')

## Вариант B — Qwen VL

In [ ]:
import base64
from openai import OpenAI

client = OpenAI(
    api_key=QWEN_API_KEY,
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
)

def describe_frame_qwen(image_path: str) -> dict:
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model='qwen-vl-max',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': 'Опиши объекты. JSON: {"objects": [...], "scene": "..."}'},
        ]}],
        max_tokens=512,
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except Exception:
        return {'raw': resp.choices[0].message.content}
